In [17]:
from langgraph.graph import StateGraph, START, END
from RAG.types import state
from RAG.llm.model import get_OpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.pydantic_v1 import BaseModel, Field
from langchain.tools import tool
from langgraph.prebuilt import ToolNode, tools_condition

In [18]:
llm = get_OpenAI()
output_parser = StrOutputParser()

In [19]:
from pydantic import BaseModel
from typing_extensions import Annotated, Sequence, TypedDict
from langchain.schema import BaseMessage

In [20]:
from RAG.tools.call_Vehicle_tools import get_brand_list_tool, get_model_list_tool, get_year_list_tool, get_detail_list_tool, get_option_list_tool, HumanRequest, State, Vehicle

In [21]:
tools = [get_brand_list_tool, get_model_list_tool, get_year_list_tool, get_detail_list_tool, get_option_list_tool, HumanRequest]

In [22]:
import os
from dotenv import load_dotenv
load_dotenv()

access_token = os.getenv("CODEF_ACCESS_TOKEN")

In [23]:
# from langgraph.graph.message import add_messages

# class State(TypedDict):
#     # 메시지 목록
#     messages: Annotated[list, add_messages]
#     # 사람에게 질문할지 여부를 묻는 상태 추가
#     ask_human: bool

In [24]:
from langgraph.graph import MessagesState
from langchain.schema import BaseMessage, HumanMessage

In [25]:
headers: headers={
    "Authorization": f"Bearer {access_token}",
    "Content-Type": "application/json"
}

State: MessagesState = {
    "messages": [HumanMessage(content="브랜드 조회해줘.")], 
    "headers":headers,
    "vehicle":Vehicle(),
    "ask_human": False
}
# config 설정
config = {"configurable": {"thread_id": "1"}}

In [26]:
llm_with_tools = llm.bind_tools(tools)

In [27]:
def call_model(state: State):
    # LLM 도구 호출을 통한 응답 생성
    response = llm_with_tools.invoke(state["messages"])

    # 사람에게 질문할지 여부 초기화
    ask_human = False

    # 도구 호출이 있고 이름이 'HumanRequest' 인 경우
    if response.tool_calls and response.tool_calls[0]["name"] == HumanRequest.__name__:
        ask_human = True

    
    # 메시지와 ask_human 상태 반환
    state.Vehicle.brand.append(response)
    
    
    return {"messages": [response], "ask_human": ask_human}

In [28]:
chatbot(State)

{'messages': [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_RCOdHqjhiadUHDoNdz6zRpML', 'function': {'arguments': '{}', 'name': 'get_brand_list'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 12, 'prompt_tokens': 388, 'total_tokens': 400, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_0aa8d3e20b', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-f3d7ac22-bb99-4ff5-bf2c-708801448f55-0', tool_calls=[{'name': 'get_brand_list', 'args': {}, 'id': 'call_RCOdHqjhiadUHDoNdz6zRpML', 'type': 'tool_call'}], usage_metadata={'input_tokens': 388, 'output_tokens': 12, 'total_tokens': 400, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})],
 